### RAG Pipelines - Data Ingestion to VectorDB Pipelinne

In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\swatis\AppData\Local\Temp\ipykernel_21128\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [10]:
### Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    """Proccess all PDF files in directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all pdfs recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages from {pdf_file.name}")
        except Exception as e:
            print(f"  ✗ Error processing {pdf_file.name}: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the specified directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: fewshot ensemble.pdf
  ✓ Loaded 9 pages from fewshot ensemble.pdf

Processing: GAN paper.pdf
  ✓ Loaded 9 pages from GAN paper.pdf

Processing: NIPS-2017-attention-is-all-you-need-Paper.pdf
  ✓ Loaded 11 pages from NIPS-2017-attention-is-all-you-need-Paper.pdf

Total documents loaded: 29


In [11]:
# The above code is same as:

# from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
# dir_loader = DirectoryLoader(
#     "../data/pdf",
#     glob="**/*.pdf", # pattern to match files
#     loader_cls=PyMuPDFLoader, # loader class to use
#     show_progress=False
# )
# pdf_documents=dir_loader.load()
# pdf_documents

# But here, we have more control as we manually created it and can write any thing like metadata, filtering, 
# handle errors gracefully, log process, etc.

# Both approaches produce the same type of output — a list of LangChain Document objects
# so you can chunk the documents in either case.

In [12]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2023-08-10T16:08:44+00:00', 'author': 'Sai Yang, Fan Liu, Delong Chen, Jun Zhou', 'keywords': 'Computer Vision: CV: Recognition (object detection & categorization), Computer Vision: CV: Transfer & low-shot & semi- and un- supervised learning', 'moddate': '2023-08-10T16:08:47+00:00', 'subject': 'Paper accepted and presented at IJCAI-2023', 'title': 'Few-shot Classification via Ensemble Learning with Multi-Order Statistics', 'source': '..\\data\\pdf\\fewshot ensemble.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'fewshot ensemble.pdf', 'file_type': 'pdf'}, page_content='Few-shot Classification via Ensemble Learning with Multi-Order Statistics\nSai Yang1 , Fan Liu2,3∗ , Delong Chen2 and Jun Zhou4\n1School of Electrical Engineering, Nantong University, Nantong, China\n2College of Computer and Information, Hohai University, Nanjing, China\n3Science and Technology on Underwater Vehicle Techno

In [13]:
### Text splitting into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [14]:
chunks = split_documents(all_pdf_documents)
chunks

Split 29 documents into 147 chunks

Example chunk:
Content: Few-shot Classification via Ensemble Learning with Multi-Order Statistics
Sai Yang1 , Fan Liu2,3∗ , Delong Chen2 and Jun Zhou4
1School of Electrical Engineering, Nantong University, Nantong, China
2Co...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2023-08-10T16:08:44+00:00', 'author': 'Sai Yang, Fan Liu, Delong Chen, Jun Zhou', 'keywords': 'Computer Vision: CV: Recognition (object detection & categorization), Computer Vision: CV: Transfer & low-shot & semi- and un- supervised learning', 'moddate': '2023-08-10T16:08:47+00:00', 'subject': 'Paper accepted and presented at IJCAI-2023', 'title': 'Few-shot Classification via Ensemble Learning with Multi-Order Statistics', 'source': '..\\data\\pdf\\fewshot ensemble.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'fewshot ensemble.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2023-08-10T16:08:44+00:00', 'author': 'Sai Yang, Fan Liu, Delong Chen, Jun Zhou', 'keywords': 'Computer Vision: CV: Recognition (object detection & categorization), Computer Vision: CV: Transfer & low-shot & semi- and un- supervised learning', 'moddate': '2023-08-10T16:08:47+00:00', 'subject': 'Paper accepted and presented at IJCAI-2023', 'title': 'Few-shot Classification via Ensemble Learning with Multi-Order Statistics', 'source': '..\\data\\pdf\\fewshot ensemble.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'fewshot ensemble.pdf', 'file_type': 'pdf'}, page_content='Few-shot Classification via Ensemble Learning with Multi-Order Statistics\nSai Yang1 , Fan Liu2,3∗ , Delong Chen2 and Jun Zhou4\n1School of Electrical Engineering, Nantong University, Nantong, China\n2College of Computer and Information, Hohai University, Nanjing, China\n3Science and Technology on Underwater Vehicle Techno

### Embedding and VectorStoreDB
faiss-cpu, chromadb are vectordb

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
# NumPy – Efficient numerical computing library for arrays, matrices, and vector operations.
# SentenceTransformer – Converts text into dense vector embeddings (384 dimensional vectors) for semantic search.
# ChromaDB – Vector database used to store, index, and retrieve embeddings efficiently.
# Settings – Configures ChromaDB behavior (e.g., persistence, telemetry, storage).
# UUID – Generates unique IDs for documents or chunks stored in the vector database.
# Typing – Provides type hints (`List`, `Dict`, `Tuple`, `Any`) to improve code readability and IDE support.
# Cosine Similarity – Measures similarity between two embedding vectors based on the angle between them.

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True) # converting text to embeddings
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2054.98it/s]


Model loaded successfully. Embedding dimension: 384


In [18]:
# 'all-MiniLM-L6-v2' is a pre-trained sentence embedding model from Hugging Face
# (developed by the Sentence Transformers project). It converts text into
# dense vector embeddings that capture semantic meaning. The model is small,
# fast, and efficient, making it well-suited for semantic search, text
# similarity, clustering, and retrieval-augmented generation (RAG) applications.

<!-- The above code is same as:
## load all pdf files from the directory
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", # pattern to match files
    loader_cls=PyMuPDFLoader, # loader class to use
    show_progress=False
)
pdf_documents=dir_loader.load()
pdf_documents

But here, we have more control as we manually created it and can write any thing like metadata, filtering, handles errors gracefully, log process, etc. -->

### Vector Store

In [19]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG", 
                        #   "hnsw:space": "cosine"
                        }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 294


In [20]:
# We are stoing the embeddings in a persistent vector store (ChromaDB).
# The vector store is initialized with a collection name and a directory to persist the data.
# We store vector_store in data directory on hard disk so that it can be reused later without recomputing embeddings.

In [21]:
### Convert text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 147 texts...


Batches: 100%|██████████| 5/5 [00:08<00:00,  1.63s/it]


Generated embeddings with shape: (147, 384)
Adding 147 documents to vector store...
Successfully added 147 documents to vector store
Total documents in collection: 441


### RAG Retriever Pipeline

In [22]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [23]:
rag_retriever

In [24]:
# This retriever is just kind of interface connected to vector store and embedding manager. 
# It takes a query, generates its embedding, and retrieves the most 
# relevant documents from the vector store based on similarity scores.

In [25]:
rag_retriever.retrieve("What is attention is all you need")
# The entire output is your context for the query, to feed into a language model for generating answers.

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 34.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_0be6f067_129',
  'content': 'convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the appendix. Not only do individual attention\nheads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic\nand semantic structure of the sentences.\n5 Training\nThis section describes the training regime for our models.\n5.1 Training Data and Batching\nWe trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million\nsentence pairs. Sentences were encoded using byte-pair encoding [ 3], which has a shared source-\ntarget vocabulary of about 37000 tokens. For English-French, we used the signiﬁcantly larger WMT\n2014 English-French dataset consisting of 36M sentences and split tokens

In [26]:
results = vectorstore.collection.query(
    query_embeddings=[
        embedding_manager.generate_embeddings(
            ["What is attention is all you need?"]
        )[0].tolist()
    ],
    n_results=5
)

print("Distances:")
print(results["distances"][0])

print("\nSources:")
for metadata in results["metadatas"][0]:
    print(metadata.get("source_file"))

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.15it/s]

Generated embeddings with shape: (1, 384)
Distances:
[0.5535237193107605, 0.5535237193107605, 0.5535237193107605, 0.5624343156814575, 0.5624343156814575]

Sources:
NIPS-2017-attention-is-all-you-need-Paper.pdf
NIPS-2017-attention-is-all-you-need-Paper.pdf
NIPS-2017-attention-is-all-you-need-Paper.pdf
NIPS-2017-attention-is-all-you-need-Paper.pdf
NIPS-2017-attention-is-all-you-need-Paper.pdf


In [27]:
rag_retriever.retrieve("Kingma and Welling")

Retrieving documents for query: 'Kingma and Welling'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.15it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8a8d9fa1_68',
  'content': 'Our work backpropagates derivatives through generative processes by using the observation that\nlim\nσ→0\n∇xEϵ∼N (0,σ2I)f (x +ϵ) =∇xf (x).\nWe were unaware at the time we developed this work that Kingma and Welling [18] and Rezende\net al. [23] had developed more general stochastic backpropagation rules, allowing one to backprop-\nagate through Gaussian distributions with ﬁnite variance, and to backpropagate to the covariance\nparameter as well as the mean. These backpropagation rules could allow one to learn the condi-\ntional variance of the generator, which we treated as a hyperparameter in this work. Kingma and\nWelling [18] and Rezende et al. [23] use stochastic backpropagation to train variational autoen-\ncoders (V AEs). Like generative adversarial networks, variational autoencoders pair a differentiable\ngenerator network with a second neural network. Unlike generative adversarial networks, the sec-',
  'metadata': {'firstpage': '2672',


### Integration VectorDB Context pipeline with LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

# Inialize the Groq LLM with your API key from environment variables
groq_api_key = "xyz"  # Replace with your actual API key or use os.getenv("GROQ_API_KEY") if stored in .env

llm = ChatGroq(api_key=groq_api_key, model="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

# Simple RAG function: retrieve context + generation
def rag_simple(query, retriever, llm, top_k=3):
    # retrieve the context
    result = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in result]) if result else ""
    if not context:
        return "No relevant context found for the query."
    # generate the answer using the Groq LLM
    prompt = f"""Use the following context to answer the question concisely.
        Context:{context}
        Question: {query}
        Answer:"""

    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content

In [37]:
answer = rag_simple("What is attention mechanism?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.82it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The attention mechanism is a component (often a self‑attention layer) that lets a model dynamically weight and combine information from different positions in a sequence, focusing on the most relevant tokens when producing each output. It computes attention scores that indicate how much each token should contribute to the representation of another token, enabling the model to capture syntactic and semantic relationships.


### Enhanced RAG Pipeline

In [39]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is attention mechanism?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.18it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The attention mechanism is a component (often a self‑attention layer) that lets a model dynamically weight and combine information from different positions in a sequence, focusing on the most relevant tokens when computing each representation. It computes similarity scores between elements, normalizes them into attention weights, and uses those weights to aggregate contextual information.
Sources: [{'source': 'NIPS-2017-attention-is-all-you-need-Paper.pdf', 'page': 6, 'score': 0.46129679679870605, 'preview': 'convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the a...'}, {'source': 'NIPS-2017-attention-is-all-you-need-Paper.pdf', 'page': 6, 'score': 0.46129679679870605, 'preview': 'convolution is equal to the combination of a self-atte

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is salman'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.62it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
In the proposed adversarial nets framework, the generative model is pitted against an adversary: a
discriminative model that learns to determine whether a sample is from the model distribution or the
data distribution. The generative model can be thou

ght of as analogous to a team of counterfeiters,
trying to produce fake currency and use it without detection, while the discriminative model is
analogous to the police, trying to detect the counterfeit currency. Competition in this game drives
both teams to improve their methods until the counterfeits are indistiguishable from the genuine
articles.
∗Ian Goodfellow is now a research scientist at Google, but did this work earlier as a UdeM student
†Jean Pouget-Abadie did this work while visiting Universit´e de Montr´eal from Ecole Polytechnique.
‡Sherjil Ozair is visiting Universit´e de Montr´eal from Indian Institute of Technology Delhi
§Yoshua Bengio is a CIFAR Senior Fellow.

In the proposed adversarial nets framework, the generative model is pitted against an adversary: a
discriminative model that learns to determine whether a sample is from the model distribution or the
data distribution. The generative model can be thought of as analogous to a team of counterfeiters,
trying to pro